# XSe2 example

This notebook demonstrates a single UppASD simulation for XSe2 materials (X=Cr, Mn, Fe)

## Scientific objective
- Run a single finite-temperature simulation with controlled J1 + K
- Compute and visualize the resulting magnetic state from `restart._UppASD_.out`

## Learning goals
- Build and run the model with `SpinSystem`, `ExchangeShellTable`, and `AnisotropyTable`
- Tune simulation parameters to find the ground state
- Produce and analyze the magnetic structure with included plotting helpers


In [ ]:
%%capture
import os

# Install needed packages
!pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple uppasd
!pip install pyvista[jupyter]


In [ ]:
# Imports and basic setup (restored from reference notebook)
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyvista as pv

# Core UppASD classes for model setup, execution, and results parsing
from uppasd.core.system import SpinSystem
from uppasd.core.exchange import ExchangeShellTable
from uppasd.core.anisotropy import AnisotropyTable
from uppasd.input.inputdata import ASDInput
from uppasd.run.simulator import ASDWorkspace, UppASDSimulator
from uppasd.core.results import ASDResults
from uppasd import set_temperature_token, validate_temperature_token, run_temperature_sweep

print("✅ Imports restored. Ensure you run cells that define `system`, `exchange`, and `inp` before preparing the workspace.")


## Step 1 — Define the crystal and initial magnetic state

We start from a minimal model to keep the physics and code path transparent.

### What each object means
- `cell`: lattice vectors that define periodic geometry
- `positions`: atom coordinates in the unit cell
- `species`: integer species labels (important for multi-component systems)
- `moments`: initial spin vectors per atom

Here we initialize moments along `+z`, which gives a simple reference state for observing relaxation behavior.

In [ ]:
# Set up a single unit cell with one atom
a = 1.0

# Hexagonal unit cell
#  1.0    0.0     0.0
# -0.5 sqrt(3)/2  0.0
#  0.0    0.0    sqrt(2)
cell = a * np.array([[1.0 , 0.0, 0.0], [-0.5 , np.sqrt(3.0)/2.0, 0.0],[0.0 , 0.0, np.sqrt(2.0)]])

# Positions in lattice coordinates
# 0.0 0.0 0.0
# 0.5 0.0 0.0
# 0.0 0.5 0.0
# 0.5 0.5 0.0
positions = np.array([[0.0 , 0.0, 0.0], [0.5 , 0.0, 0.0], [0.0 , 0.5, 0.0], [0.5 , 0.5, 0.0]])
# Transform to cartesian coordinates:
positions = positions @ cell.T
natom = positions.shape[0]
species = np.ones(natom, dtype=int)

# Define magnetic moments
m_FeSe2 = 3.593
m_MnSe2 = 3.685
m_CrSe2 = 2.753

# Setup moments
moments = np.zeros((natom, 3))
#moments[:, 2] = 1.0
moments[:, 2] = m_FeSe2

system = SpinSystem(cell, positions, species, moments)

## Step 2 — Define exchange interactions

The `ExchangeShellTable` encodes Heisenberg interactions shell-by-shell.

In the next code cell we define a frustrated J1 system with uniaxial anisotropy K. The exchange interactions are material specific and comes from the MIDB database https://aphys-midb.pdc.kth.se

The systems considered here are monolayers of $2H-XSe_2$ where $X=(Fe, Cr, Mn)$

In [ ]:
exchange = ExchangeShellTable()

# Definition of exchange interactions
J1_FeSe2 = -0.9621860632143219
J1_MnSe2 = -0.15209307331736713
J1_CrSe2 = -0.6404207391737375

# Exchange couplings J_ij in mRy (shell-resolved)
J_ij = [J1_FeSe2]
r_ij = np.array([[a/2.0, 0, 0]])

for ij, jij in enumerate(J_ij):
    if abs(jij)>0.0:
        exchange.add_bond(1, 1, ij+1, r_ij[ij], jij)

In [ ]:
# Step 2b — Define on-site anisotropy (kfile)
# Create an anisotropy table and add a uniaxial anisotropy on atom 1:
# Format: add_site(iatom, ani_type, K2, K4, direction)
# ani_type=1 for uniaxial, K2<0 for easy-axis

# Definition of anisotropies
K_FeSe2 = 0.1267115617468
K_MnSe2 = -0.286928923756659
K_CrSe2 = -0.094909458700267

# Set the anisotropy coupling
anisotropy = AnisotropyTable()
for i_site in range(natom):
    anisotropy.add_site(i_site + 1, 1, K_FeSe2, 0.0, [0.0, 0.0, 1.0])


## Step 3 — Configure input blocks

`ASDInput` organizes UppASD keywords by conceptual blocks (`system`, `initial`, `simulation`, `measurables`).

We use an LLG initial phase (`ip_mode='S'`) with scalar syntax:
- `ip_temp`: initial-phase temperature
- `ip_nstep`: number of initial-phase steps
- `ip_timestep`, `ip_damping`: integration controls

Then we set production simulation controls and request output needed for time-series analysis (`averages` and `totenergy`).

### Note:
In this particular example one needs to be cautions to relax/initialize the system carefully, since it is frustrated and anisotropic.

**Import & setup:** An imports/setup cell has been added above — run it first. If you have previously defined `system`, `exchange`, and `inp`, the workspace preparation cell will initialize the run directory.

In [ ]:
# Define simulation and measurement parameters (ASDInput)
inp = ASDInput()
Nx = 32
Ny = 32
Nz = 1

# Define a temperature to ensure same value is given in initial and measurement phases (optional)
Temperature = 0
Temperature_shift = 5.0

inp.block("system").set(
    ncell=(Nx, Ny, Nz),
    bc=("P", "P", "0"),
    do_prnstruct=2,
    sym=3,
    posfiletype='D',
 )

# Initial phase settings
inp.block("initial").set(ip_mode="H", ip_temp=5.0, ip_mcnstep=5000, initmag=1)

# Main simulation settings
inp.block("simulation").set(mode="S", nstep=5000, timestep=1e-15, damping=0.5, temperature=5.0)

# Output settings needed for thermodynamic analysis
inp.block("measurables").set(plotenergy=1, do_avrg="Y", do_cumu="Y")

# Synchronize initial-phase and simulation temperature tokens
set_temperature_token(inp, 5.0)
print(validate_temperature_token(inp))

## Run the temperature sweep

`run_temperature_sweep(...)` processes one temperature at a time and prints progress in the notebook output.

Tip: start with a coarse temperature grid for quick testing, then refine near the transition region.

In [ ]:
# Temperature sweep example using built-in helper (with progress output)
temperatures = np.linspace(10.0, 150.0, 15)

sweep_results = run_temperature_sweep(
    workdir_root="XSe2_sweep",
    temperatures=temperatures,
    system=system,
    inp=inp,
    exchange=exchange,
    ani=anisotropy,
    dmi=None,
    clean=True,
    verbose=True,
 )

sweep_df = pd.DataFrame(sweep_results)
display(sweep_df)

## Step 5 — Analyze thermodynamic curves and estimate transition temperatures

The plots below provide complementary transition indicators:
- `m(T)`: order parameter decay with temperature
- Binder cumulant: finite-size-sensitive shape/crossing behavior
- `chi(T)`: magnetic fluctuation peak
- `cv(T)`: energy fluctuation peak
- `dE/dT`: additional numerical trend from energy curve

### Interpretation notes
- Peak positions from `chi` and `cv` are useful first estimates of transition windows.
- For publishable phase boundaries, repeat for larger system sizes and perform finite-size scaling.

In [ ]:
# Phase-diagram diagnostics from temperature sweep (units: T[K], m[μ_B], E[mRy])
required = ["temperature", "m", "binder", "chi", "cv", "energy"]
missing = [col for col in required if col not in sweep_df.columns]
if missing:
    raise ValueError(f"Missing columns in sweep_df: {missing}")

plot_df = sweep_df.dropna(subset=required).sort_values("temperature").reset_index(drop=True)
if plot_df.empty:
    raise ValueError("No valid sweep rows to plot")

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes = axes.ravel()

axes[0].plot(plot_df["temperature"], plot_df["m"], marker="o")
axes[0].set_title("Magnetization")
axes[0].set_xlabel("Temperature [K]")
axes[0].set_ylabel("m [μ_B]")

axes[1].plot(plot_df["temperature"], plot_df["binder"], marker="o")
axes[1].set_title("Binder Cumulant")
axes[1].set_xlabel("Temperature [K]")
axes[1].set_ylabel("U4 [dimensionless]")

axes[2].plot(plot_df["temperature"], plot_df["chi"], marker="o")
axes[2].set_title("Susceptibility")
axes[2].set_xlabel("Temperature [K]")
axes[2].set_ylabel("chi [UppASD units]")

axes[3].plot(plot_df["temperature"], plot_df["cv"], marker="o")
axes[3].set_title("Specific Heat")
axes[3].set_xlabel("Temperature [K]")
axes[3].set_ylabel("cv [UppASD units]")

axes[4].plot(plot_df["temperature"], plot_df["energy"], marker="o")
axes[4].set_title("Energy")
axes[4].set_xlabel("Temperature [K]")
axes[4].set_ylabel("E [mRy]")

# Optional numerical derivative dE/dT as additional transition indicator
dEdT = np.gradient(plot_df["energy"].to_numpy(), plot_df["temperature"].to_numpy())
axes[5].plot(plot_df["temperature"], dEdT, marker="o")
axes[5].set_title("dE/dT")
axes[5].set_xlabel("Temperature [K]")
axes[5].set_ylabel("dE/dT [mRy/K]")

plt.show()

# Simple transition-temperature indicators
tc_chi = float(plot_df.loc[plot_df["chi"].idxmax(), "temperature"])
tc_cv = float(plot_df.loc[plot_df["cv"].idxmax(), "temperature"])
print(f"Estimated Tc from chi peak: {tc_chi:.3f} K")
print(f"Estimated Tc from cv peak : {tc_cv:.3f} K")